# Case Study: Titanic — Identificação Visual de MCAR, MAR e MNAR com MissingData

Este estudo de caso demonstra como as visualizações do pacote **MissingData** ajudam pessoas não técnicas (médicos, gestores, analistas) a reconhecer **padrões de dados em falta**.
O foco não é treinar modelos, mas sim **aprender a interpretar gráficos** e inferir o mecanismo de missingness.

**Porque é que dados em falta são um problema real?**
1. Podem enviesar resultados quando certos perfis aparecem menos no conjunto de dados.
2. O tratamento adequado depende da **causa** das faltas.
3. Assumir MCAR quando o mecanismo é MAR ou MNAR pode levar a conclusões erradas.

Neste notebook simulamos **30% de missingness** em três cenários (MCAR, MAR, MNAR) e comparamos sempre duas versões do dataset:
- **Original completo** (sem missing values)
- **Versão com missingness**

## Os três mecanismos, em linguagem simples

- **MCAR**: as faltas acontecem ao acaso, sem depender de nenhuma variável.
- **MAR**: as faltas dependem de uma variável observável.
- **MNAR**: as faltas dependem do próprio valor que está a faltar.

Em dados reais, raramente sabemos o mecanismo com certeza. Aqui sabemos porque estamos a simular.

## O dataset Titanic e por que é útil

O dataset Titanic descreve passageiros de um naufrágio histórico e inclui variáveis intuitivas como idade, classe, tarifa e familiares a bordo.
É útil para ensino porque permite **interpretar padrões de forma imediata**, sem jargão estatístico.

## Configuração

In [ ]:
# Se estiveres num ambiente limpo, podes instalar dependências:
# %pip install -q seaborn pyampute

from typing import Optional, List

import numpy as np
import pandas as pd
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pyampute.ampute import MultivariateAmputation

from missingfcup.core.MissingData import MissingData
from missingfcup.plots.Panel import Panel

## Preparação de um dataset "ground truth"

Para sabermos exatamente o mecanismo, removemos os NaNs originais e partimos de um dataset completo.
Criamos também variáveis derivadas simples para enriquecer a análise:
- `family_size` = sibsp + parch + 1
- `fare_per_person` = fare / family_size
- `sex_code` = 1 para homens, 0 para mulheres (variável numérica observável)

Nota: `seaborn.load_dataset("titanic")` pode precisar de acesso à internet.

In [ ]:
_df_raw = sns.load_dataset("titanic")

# Ground truth completo
# Removemos NaNs originais para garantir uma base sem missingness
df_complete = _df_raw.dropna().reset_index(drop=True)

# Variáveis derivadas (contínuas e intuitivas)
df_complete["family_size"] = df_complete["sibsp"] + df_complete["parch"] + 1
df_complete["fare_per_person"] = df_complete["fare"] / df_complete["family_size"]

# Variáveis numéricas observáveis adicionais
# sex_code = 1 (male), 0 (female)
df_complete["sex_code"] = (df_complete["sex"] == "male").astype(int)

print("Dimensões do dataset completo:", df_complete.shape)

# Mostramos apenas algumas colunas para contexto
cols_preview = ["survived", "sex", "age", "pclass", "fare", "sibsp", "parch"]
df_complete[cols_preview].head()

## Funções auxiliares (simulação e visualização)

Usamos **pyampute** para gerar missingness controlada. A função abaixo amputa apenas a coluna alvo e devolve um novo DataFrame.

In [ ]:
PRESENT_COLOR = "#2ca02c"  # verde
MISSING_COLOR = "#d62728"  # vermelho
RATE_COLORSCALE = [[0.0, PRESENT_COLOR], [1.0, MISSING_COLOR]]

# Lista explícita de colunas numéricas usadas na simulação
NUMERIC_COLS = [
    "age",
    "fare",
    "sibsp",
    "parch",
    "pclass",
    "family_size",
    "fare_per_person",
    "sex_code",
    "survived",
]


def ampute_with_pyampute(
    df: pd.DataFrame,
    target_col: str,
    mechanism: str,
    driver_col: Optional[str] = None,
    prop: float = 0.30,
    seed: int = 42,
    score_func: str = "SIGMOID-RIGHT",
    numeric_cols: Optional[List[str]] = None,
) -> pd.DataFrame:
    # Aplica missingness numa única coluna usando pyampute
    if numeric_cols is None:
        raise ValueError("numeric_cols deve ser fornecido explicitamente.")

    mechanism = mechanism.upper()
    numeric_df = df[numeric_cols].copy()

    required = [target_col] + ([driver_col] if driver_col else [])
    for col in required:
        if col not in numeric_df.columns:
            raise ValueError(f"Coluna não encontrada nos dados numéricos: {col}")

    if mechanism == "MCAR":
        patterns = [
            {
                "incomplete_vars": [target_col],
                "mechanism": "MCAR",
            }
        ]
    elif mechanism == "MAR":
        if driver_col is None:
            raise ValueError("driver_col é obrigatório para MAR")
        patterns = [
            {
                "incomplete_vars": [target_col],
                "mechanism": "MAR",
                "weights": {driver_col: 1.0},
                "score_to_probability_func": score_func,
            }
        ]
    elif mechanism == "MNAR":
        patterns = [
            {
                "incomplete_vars": [target_col],
                "mechanism": "MNAR",
                "weights": {target_col: 1.0},
                "score_to_probability_func": score_func,
            }
        ]
    else:
        raise ValueError(f"Mecanismo desconhecido: {mechanism}")

    amputer = MultivariateAmputation(prop=prop, patterns=patterns, seed=seed)
    amputed_numeric = amputer.fit_transform(numeric_df)

    df_out = df.copy()
    df_out[target_col] = amputed_numeric[target_col]
    return df_out


def compare_panel(title: str, left_plot, right_plot) -> None:
    # Mostra dois plots MissingData lado a lado
    panel = Panel(
        plots=[left_plot, right_plot],
        title=title,
        max_cols=2,
    )
    panel.show()


def scatter_missing_indicator_side_by_side(
    df_left: pd.DataFrame,
    df_right: pd.DataFrame,
    x: str,
    y: str,
    missing_col: str,
    title_left: str,
    title_right: str,
    overall_title: str,
) -> None:
    # Scatterplot com cores definidas por is_missing(missing_col)
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[title_left, title_right],
        horizontal_spacing=0.12,
    )

    for idx, df in enumerate([df_left, df_right]):
        col = 1 if idx == 0 else 2
        missing_mask = df[missing_col].isna()

        # Pontos presentes
        present = df.loc[~missing_mask]
        fig.add_trace(
            go.Scatter(
                x=present[x],
                y=present[y],
                mode="markers",
                name="Presente" if idx == 1 else None,
                marker=dict(color=PRESENT_COLOR, size=7, opacity=0.65),
                showlegend=(idx == 1),
            ),
            row=1,
            col=col,
        )

        # Pontos em falta
        missing = df.loc[missing_mask]
        fig.add_trace(
            go.Scatter(
                x=missing[x],
                y=missing[y],
                mode="markers",
                name="Missing" if idx == 1 else None,
                marker=dict(color=MISSING_COLOR, size=7, opacity=0.75, symbol="x"),
                showlegend=(idx == 1),
            ),
            row=1,
            col=col,
        )

    fig.update_xaxes(title_text=x, row=1, col=1)
    fig.update_xaxes(title_text=x, row=1, col=2)
    fig.update_yaxes(title_text=y, row=1, col=1)
    fig.update_yaxes(title_text=y, row=1, col=2)

    fig.update_layout(
        title_text=overall_title,
        width=1400,
        height=520,
        legend_title_text="Estado",
    )
    fig.show()


def missing_indicator_correlations(
    df: pd.DataFrame,
    missing_col: str,
    numeric_cols: List[str],
) -> pd.Series:
    # Calcula correlações entre is_missing(missing_col) e variáveis numéricas
    indicator = df[missing_col].isna().astype(int).rename("is_missing")
    corr = pd.concat([indicator, df[numeric_cols]], axis=1).corr()
    return corr.loc["is_missing", numeric_cols]


def correlation_heatmap_side_by_side(
    series_left: pd.Series,
    series_right: pd.Series,
    title_left: str,
    title_right: str,
    overall_title: str,
) -> None:
    # Heatmap 1xN com correlações, lado a lado
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[title_left, title_right],
        horizontal_spacing=0.12,
    )

    for idx, series in enumerate([series_left, series_right]):
        col = 1 if idx == 0 else 2
        values = series.values
        text = ["N/A" if pd.isna(v) else f"{v:.2f}" for v in values]
        z = pd.Series(values).fillna(0).to_numpy()

        fig.add_trace(
            go.Heatmap(
                z=[z],
                x=series.index.tolist(),
                y=["corr"],
                colorscale="RdBu",
                zmin=-1,
                zmax=1,
                zmid=0,
                text=[text],
                texttemplate="%{text}",
                showscale=(idx == 1),
                colorbar=dict(title="Correlação"),
            ),
            row=1,
            col=col,
        )

    fig.update_yaxes(showticklabels=False)
    fig.update_layout(title_text=overall_title, width=1400, height=420)
    fig.show()

# 5. Simular MCAR

Neste cenário, **30%** dos valores de `age` faltam ao acaso, sem relação com outras variáveis.

In [ ]:
MCAR_TARGET = "age"

df_mcar = ampute_with_pyampute(
    df=df_complete,
    target_col=MCAR_TARGET,
    mechanism="MCAR",
    prop=0.30,
    seed=7,
    numeric_cols=NUMERIC_COLS,
)

md_mcar = MissingData(df_mcar)
md_original = MissingData(df_complete)

# 6. Diagnóstico visual de MCAR

### MCAR — Visualização 1 (inconclusiva): Missing count por coluna

Eixos: colunas no eixo horizontal e contagem de faltas no eixo vertical.

In [ ]:
compare_panel(
    "MCAR: Missing count por coluna (inconclusivo)",
    md_original.missing_count_barchart(
        title="Original (completo)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
    md_mcar.missing_count_barchart(
        title="Com MCAR (30% em age)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
)

**O que esperávamos ver:** apenas a coluna `age` com um número elevado de faltas.
**O que observamos:** no gráfico da direita, a barra de `age` é a única alta; no gráfico da esquerda não há barras.
**O que isto implica:** sabemos **quanto** falta, mas ainda não sabemos **por que** falta.

### MCAR — Visualização 2 (inconclusiva): Missing rate por coluna

Eixos: colunas no eixo horizontal; a cor indica a taxa de missingness.

In [ ]:
compare_panel(
    "MCAR: Missing rate por coluna (inconclusivo)",
    md_original.column_missing_rate_heatmap(
        title="Original (completo)",
        colorscale=RATE_COLORSCALE,
        show_colorbar=False,
    ),
    md_mcar.column_missing_rate_heatmap(
        title="Com MCAR (30% em age)",
        colorscale=RATE_COLORSCALE,
        show_colorbar=False,
    ),
)

**O que esperávamos ver:** a célula de `age` mais vermelha no dataset com MCAR.
**O que observamos:** a célula de `age` no gráfico da direita é a única destacada; as restantes estão verdes.
**O que isto implica:** novamente vemos **magnitude**, mas não o mecanismo.

### MCAR — Visualização 3 (diagnóstica): Heatmap de missingness

Eixos: linhas são passageiros; colunas são variáveis. Vermelho indica faltas.

In [ ]:
cols_mcar = ["age", "fare", "pclass", "sibsp", "parch"]

compare_panel(
    "MCAR: Heatmap de missingness (padrão aleatório)",
    md_original.heatmap(
        selected_columns=cols_mcar,
        title="Original (sem missing)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
    md_mcar.heatmap(
        selected_columns=cols_mcar,
        title="MCAR em age",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
)

**O que esperávamos ver:** células vermelhas dispersas, sem blocos visíveis.
**O que observamos:** olhe para a coluna `age` no gráfico da direita; o vermelho aparece espalhado ao longo de todo o eixo vertical.
**O que isto implica:** este padrão é consistente com **MCAR**, porque não há concentração em nenhum grupo.

### MCAR — Visualização 4 (diagnóstica): Scatterplot com indicador de missingness

Eixos: `fare` no eixo x e `parch` no eixo y.
Pontos vermelhos representam passageiros com `age` em falta.

In [ ]:
scatter_missing_indicator_side_by_side(
    df_left=df_complete,
    df_right=df_mcar,
    x="fare",
    y="parch",
    missing_col="age",
    title_left="Original (sem missing)",
    title_right="MCAR (missing em age)",
    overall_title="MCAR: is_missing(age) sobre fare vs parch",
)

**O que esperávamos ver:** pontos vermelhos distribuídos de forma aleatória.
**O que observamos:** repare que os pontos vermelhos estão espalhados por todo o plano, sem concentração em faixas específicas de `fare` ou `parch`.
**O que isto implica:** a ausência de padrão reforça a hipótese de **MCAR**.

### MCAR — Visualização 5 (diagnóstica): Correlação com is_missing(age)

Aqui medimos a correlação entre **is_missing(age)** e as variáveis numéricas.
Em MCAR, estas correlações devem ser próximas de zero.

In [ ]:
cor_left = missing_indicator_correlations(df_complete, "age", NUMERIC_COLS)
cor_right = missing_indicator_correlations(df_mcar, "age", NUMERIC_COLS)

correlation_heatmap_side_by_side(
    series_left=cor_left,
    series_right=cor_right,
    title_left="Original (sem missing)",
    title_right="MCAR (missing em age)",
    overall_title="MCAR: correlação entre is_missing(age) e variáveis numéricas",
)

**O que esperávamos ver:** correlações próximas de zero no dataset com MCAR.
**O que observamos:** no gráfico da direita, as cores estão perto do branco (valores próximos de 0). No gráfico da esquerda, não há missingness e a correlação é efetivamente N/A.
**O que isto implica:** este é um indício estatístico forte de **MCAR**.

# 7. Simular MAR (com múltiplas variáveis em falta)

Aqui a falta **não é aleatória**: ela pode ser explicada por variáveis observadas.
Para tornar o padrão mais claro, criamos missingness em **quatro variáveis**:

- `age` depende de `pclass`
- `fare` depende de `sex_code`
- `sibsp` depende de `pclass`
- `parch` depende de `survived`

Cada coluna tem cerca de **30% de missingness**.

In [ ]:
# Aplicar MAR em múltiplas colunas, uma de cada vez
# Cada coluna depende de uma variável observada diferente

df_mar = df_complete.copy()

df_mar = ampute_with_pyampute(
    df=df_mar,
    target_col="age",
    mechanism="MAR",
    driver_col="pclass",
    prop=0.30,
    seed=11,
    numeric_cols=NUMERIC_COLS,
)

df_mar = ampute_with_pyampute(
    df=df_mar,
    target_col="fare",
    mechanism="MAR",
    driver_col="sex_code",
    prop=0.30,
    seed=12,
    numeric_cols=NUMERIC_COLS,
)

df_mar = ampute_with_pyampute(
    df=df_mar,
    target_col="sibsp",
    mechanism="MAR",
    driver_col="pclass",
    prop=0.30,
    seed=13,
    numeric_cols=NUMERIC_COLS,
)

df_mar = ampute_with_pyampute(
    df=df_mar,
    target_col="parch",
    mechanism="MAR",
    driver_col="survived",
    prop=0.30,
    seed=14,
    numeric_cols=NUMERIC_COLS,
)

md_mar = MissingData(df_mar)

# 8. Diagnóstico visual de MAR

### MAR — Visualização 1 (inconclusiva): Missing count por coluna

In [ ]:
compare_panel(
    "MAR: Missing count por coluna (inconclusivo)",
    md_original.missing_count_barchart(
        title="Original (completo)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
    md_mar.missing_count_barchart(
        title="Com MAR (4 colunas)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
)

**O que esperávamos ver:** várias barras altas (uma por cada coluna com missingness).
**O que observamos:** no gráfico da direita, `age`, `fare`, `sibsp` e `parch` aparecem com contagens elevadas.
**O que isto implica:** sabemos **quais colunas** têm problemas, mas não **por que** têm.

### MAR — Visualização 2 (inconclusiva): Missing rate por coluna

In [ ]:
compare_panel(
    "MAR: Missing rate por coluna (inconclusivo)",
    md_original.column_missing_rate_heatmap(
        title="Original (completo)",
        colorscale=RATE_COLORSCALE,
        show_colorbar=False,
    ),
    md_mar.column_missing_rate_heatmap(
        title="Com MAR (4 colunas)",
        colorscale=RATE_COLORSCALE,
        show_colorbar=False,
    ),
)

**O que esperávamos ver:** quatro células mais vermelhas no dataset com MAR.
**O que observamos:** `age`, `fare`, `sibsp` e `parch` estão destacadas.
**O que isto implica:** o gráfico quantifica o problema, mas ainda não mostra o mecanismo.

### MAR — Visualização 3 (diagnóstica): Heatmap ordenado por `pclass`

Ordenamos por `pclass` para observar o padrão em `age` e `sibsp`.

In [ ]:
cols_mar = ["pclass", "age", "sibsp", "fare", "parch"]

compare_panel(
    "MAR: Heatmap ordenado por pclass",
    md_original.heatmap(
        selected_columns=cols_mar,
        order_by=[
            {"axis": "rows", "column": "pclass", "type": "numeric", "ascending": True}
        ],
        title="Original (sem missing)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
    md_mar.heatmap(
        selected_columns=cols_mar,
        order_by=[
            {"axis": "rows", "column": "pclass", "type": "numeric", "ascending": True}
        ],
        title="MAR (age e sibsp dependem de pclass)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
)

**O que esperávamos ver:** missingness concentrada em classes específicas.
**O que observamos:** no gráfico da direita, foque-se nos blocos de `age` e `sibsp`; o vermelho concentra-se na parte inferior (classe 3).
**O que isto implica:** missingness explicado por **variável observada**, típico de **MAR**.

### MAR — Visualização 4 (diagnóstica): Scatterplots por relação alvo–driver

Mostramos duas relações de MAR:
- `pclass` → missingness em `age`
- `sex_code` → missingness em `fare`

In [ ]:
# Relação 1: pclass -> age
compare_panel(
    "MAR: pclass influencia missingness em age",
    md_original.scatterplot(
        x="pclass",
        y="age",
        title="Original (completo)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
    md_mar.scatterplot(
        x="pclass",
        y="age",
        title="Com MAR (age depende de pclass)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
)

# Relação 2: sex_code -> fare
compare_panel(
    "MAR: sex_code influencia missingness em fare",
    md_original.scatterplot(
        x="sex_code",
        y="fare",
        title="Original (completo)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
    md_mar.scatterplot(
        x="sex_code",
        y="fare",
        title="Com MAR (fare depende de sex_code)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
)

**O que esperávamos ver:** pontos vermelhos concentrados em valores específicos dos drivers.
**O que observamos:** no gráfico de `pclass` vs `age`, repare no conjunto de pontos vermelhos em `pclass=3`. No gráfico de `sex_code` vs `fare`, os vermelhos aparecem mais num dos grupos.
**O que isto implica:** missingness explicado por variáveis observadas → **MAR**.

### MAR — Visualização 5 (diagnóstica): Missingness correlation heatmap

Esta visualização mostra como as **faltas em diferentes colunas** se relacionam entre si.

In [ ]:
compare_panel(
    "MAR: correlação de missingness entre colunas",
    md_original.missingness_correlation_heatmap(
        selected_columns=["age", "fare", "sibsp", "parch"],
        title="Original (sem missing)",
        show_values=False,
        drop_constant_columns=False,
    ),
    md_mar.missingness_correlation_heatmap(
        selected_columns=["age", "fare", "sibsp", "parch"],
        title="Com MAR (4 colunas)",
        show_values=False,
        drop_constant_columns=False,
    ),
)

**O que esperávamos ver:** algumas correlações positivas entre colunas com missingness.
**O que observamos:** no gráfico da direita, há blocos com cores mais intensas, sugerindo padrões partilhados.
**O que isto implica:** confirma que as faltas não são aleatórias e seguem relações entre variáveis.

### MAR — Visualização 6 (diagnóstica): UpSet plot

O UpSet mostra **interseções** de missingness entre várias colunas.
No dataset original não existem interseções (não há missingness), por isso o gráfico fica vazio.

In [ ]:
# Placeholder simples para o original (sem missingness)
fig_placeholder = go.Figure()
fig_placeholder.add_annotation(
    text="Original: sem missingness",
    x=0.5,
    y=0.5,
    showarrow=False,
    font=dict(size=16),
)
fig_placeholder.update_xaxes(visible=False)
fig_placeholder.update_yaxes(visible=False)
fig_placeholder.update_layout(title="Original (sem missing)", width=700, height=450)
fig_placeholder.show()

# UpSet real para o dataset com MAR
md_mar.upset_plot(
    selected_columns=["age", "fare", "sibsp", "parch"],
    title="Com MAR (4 colunas)",
    missing_color=MISSING_COLOR,
).show()

**O que esperávamos ver:** interseções relevantes entre colunas com missingness.
**O que observamos:** no gráfico do MAR, algumas combinações de colunas aparecem com maior frequência.
**O que isto implica:** há padrões de co‑ocorrência típicos de MAR.

### MAR — Visualização 7 (diagnóstica): Pattern bar chart

Este gráfico mostra **combinações frequentes** de colunas em falta.

In [ ]:
compare_panel(
    "MAR: padrões de missingness",
    md_original.pattern_barchart(
        selected_columns=["age", "fare", "sibsp", "parch"],
        value="percent",
        title="Original (sem missing)",
        missing_color=MISSING_COLOR,
    ),
    md_mar.pattern_barchart(
        selected_columns=["age", "fare", "sibsp", "parch"],
        value="percent",
        title="Com MAR (4 colunas)",
        missing_color=MISSING_COLOR,
    ),
)

**O que esperávamos ver:** padrões de combinações recorrentes no dataset com MAR.
**O que observamos:** o gráfico da direita mostra barras com percentagens relevantes em combinações específicas.
**O que isto implica:** reforça a presença de um mecanismo estruturado (não aleatório).

### MAR — Visualização 8 (diagnóstica): Parallel coordinates

As linhas vermelhas indicam missingness em `fare` (uma das variáveis MAR).

In [ ]:
compare_panel(
    "MAR: Parallel coordinates (fare depende de sex_code)",
    md_original.parallel_coordinates(
        selected_columns=["sex_code", "fare", "pclass", "age"],
        missingness_color_column="fare",
        normalize=False,
        impute_below_range_frac=0.1,
        show_colorbar=False,
        title="Original (completo)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
    md_mar.parallel_coordinates(
        selected_columns=["sex_code", "fare", "pclass", "age"],
        missingness_color_column="fare",
        normalize=False,
        impute_below_range_frac=0.1,
        show_colorbar=False,
        title="Com MAR (fare)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
)

**O que esperávamos ver:** linhas vermelhas associadas a valores específicos de `sex_code`.
**O que observamos:** no gráfico da direita, repare na concentração das linhas vermelhas num dos lados do eixo `sex_code`.
**O que isto implica:** missingness explicado por uma variável observada → **MAR**.

# 9. Simular MNAR (mais claro vs MCAR)

Neste cenário, **30%** dos valores de `fare_per_person` faltam e a probabilidade de faltar depende do próprio valor.
Para deixar a diferença mais clara face ao MCAR, vamos procurar sinais de **concentração** em valores altos (via proxy).

In [ ]:
MNAR_TARGET = "fare_per_person"

df_mnar = ampute_with_pyampute(
    df=df_complete,
    target_col=MNAR_TARGET,
    mechanism="MNAR",
    prop=0.30,
    seed=13,
    numeric_cols=NUMERIC_COLS,
)

md_mnar = MissingData(df_mnar)

# 10. Diagnóstico visual de MNAR (com raciocínio por proxy)

**Nota importante:** MNAR é o mecanismo mais difícil de confirmar visualmente.
Aqui usamos `fare` como **proxy** para `fare_per_person`, porque são variáveis correlacionadas.
Isto **não prova** MNAR; fornece apenas indícios indiretos.

### MNAR — Visualização 1 (inconclusiva): Missing count por coluna

In [ ]:
compare_panel(
    "MNAR: Missing count por coluna (inconclusivo)",
    md_original.missing_count_barchart(
        title="Original (completo)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
    md_mnar.missing_count_barchart(
        title="Com MNAR (30% em fare_per_person)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
)

**O que esperávamos ver:** apenas `fare_per_person` com faltas.
**O que observamos:** a barra de `fare_per_person` destaca-se no gráfico da direita.
**O que isto implica:** continuamos sem saber o mecanismo, apenas o volume de missingness.

### MNAR — Visualização 2 (inconclusiva): Missing rate por coluna

In [ ]:
compare_panel(
    "MNAR: Missing rate por coluna (inconclusivo)",
    md_original.column_missing_rate_heatmap(
        title="Original (completo)",
        colorscale=RATE_COLORSCALE,
        show_colorbar=False,
    ),
    md_mnar.column_missing_rate_heatmap(
        title="Com MNAR (30% em fare_per_person)",
        colorscale=RATE_COLORSCALE,
        show_colorbar=False,
    ),
)

**O que esperávamos ver:** a célula de `fare_per_person` mais vermelha no dataset com MNAR.
**O que observamos:** a intensidade confirma a taxa, mas não revela a causa.
**O que isto implica:** é preciso observar padrões indiretos.

### MNAR — Visualização 3 (diagnóstica): Scatterplot `fare` vs `fare_per_person`

Eixos: `fare` no x e `fare_per_person` no y.
Pontos vermelhos indicam `fare_per_person` em falta.

In [ ]:
compare_panel(
    "MNAR: Scatterplot fare vs fare_per_person",
    md_original.scatterplot(
        x="fare",
        y="fare_per_person",
        title="Original (completo)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
    md_mnar.scatterplot(
        x="fare",
        y="fare_per_person",
        title="Com MNAR (fare_per_person)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
)

**O que esperávamos ver:** mais pontos vermelhos associados a tarifas elevadas.
**O que observamos:** no gráfico da direita, foque-se na zona de tarifas altas (lado direito); os pontos vermelhos aparecem com maior frequência.
**O que isto implica:** há indícios de que a falta depende do próprio valor, mas esta evidência é indireta.

### MNAR — Visualização 4 (diagnóstica): Heatmap ordenado por `fare`

Ordenamos por `fare` para usar esta variável como proxy de `fare_per_person`.

In [ ]:
cols_mnar = ["fare", "fare_per_person", "pclass", "age"]

compare_panel(
    "MNAR: Heatmap ordenado por fare (proxy)",
    md_original.heatmap(
        selected_columns=cols_mnar,
        order_by=[
            {"axis": "rows", "column": "fare", "type": "numeric", "ascending": False}
        ],
        title="Original (sem missing)",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
    md_mnar.heatmap(
        selected_columns=cols_mnar,
        order_by=[
            {"axis": "rows", "column": "fare", "type": "numeric", "ascending": False}
        ],
        title="MNAR em fare_per_person",
        missing_color=MISSING_COLOR,
        present_color=PRESENT_COLOR,
    ),
)

**O que esperávamos ver:** missingness concentrada em linhas com `fare` alto.
**O que observamos:** no gráfico da direita, olhe para a parte superior (tarifas mais altas); a coluna `fare_per_person` tem mais vermelho.
**O que isto implica:** isto sugere MNAR **via proxy**, mas não é prova definitiva.

### MNAR — Visualização 5 (diagnóstica): Correlação com is_missing(fare_per_person)

Aqui medimos a correlação entre **is_missing(fare_per_person)** e variáveis numéricas.
Isto ajuda a distinguir MNAR de MCAR.

In [ ]:
cor_left_mnar = missing_indicator_correlations(df_complete, "fare_per_person", NUMERIC_COLS)
cor_right_mnar = missing_indicator_correlations(df_mnar, "fare_per_person", NUMERIC_COLS)

correlation_heatmap_side_by_side(
    series_left=cor_left_mnar,
    series_right=cor_right_mnar,
    title_left="Original (sem missing)",
    title_right="MNAR (fare_per_person)",
    overall_title="MNAR: correlação entre is_missing(fare_per_person) e variáveis numéricas",
)

**O que esperávamos ver:** correlações diferentes de zero em variáveis correlacionadas com `fare_per_person`.
**O que observamos:** no gráfico da direita, repare que `fare` apresenta uma correlação mais intensa com o indicador de missingness.
**O que isto implica:** ao contrário do MCAR, há um padrão associado a variáveis relacionadas com o valor em falta.

### MNAR — Visualização 6 (diagnóstica): Scatterplot com indicador de missingness

Aqui não usamos a variável em falta no eixo, para evitar ambiguidade visual.

In [ ]:
scatter_missing_indicator_side_by_side(
    df_left=df_complete,
    df_right=df_mnar,
    x="fare",
    y="pclass",
    missing_col="fare_per_person",
    title_left="Original (sem missing)",
    title_right="MNAR (missing em fare_per_person)",
    overall_title="MNAR: is_missing(fare_per_person) sobre fare vs pclass",
)

**O que esperávamos ver:** pontos vermelhos concentrados em tarifas altas.
**O que observamos:** no gráfico da direita, foque-se no lado direito (tarifas mais altas); os pontos vermelhos concentram-se ali.
**O que isto implica:** padrão consistente com MNAR, e claramente diferente do espalhamento aleatório de MCAR.

### MNAR vs MAR — Evitar confusão

É fácil confundir MAR com MNAR quando usamos proxies.
- **MAR**: a falta depende de outra variável observada.
- **MNAR**: a falta depende do próprio valor que está em falta.

Aqui usamos `fare` como proxy apenas porque está correlacionado com `fare_per_person`.
Isto **não transforma** o mecanismo em MAR; serve apenas como evidência indireta.

# 11. Fluxo de diagnóstico para profissionais

Um roteiro simples para quem não é especialista:
1. Comece com **missing count** e **missing rate** para localizar colunas problemáticas.
2. Use **heatmaps ordenados** para ver se o missingness se concentra em grupos específicos.
3. Faça **scatterplots** e **parallel coordinates** para confirmar padrões visuais.
4. Use **missingness correlation**, **UpSet** e **Pattern Bar Chart** quando várias colunas estão em falta.
5. Se suspeitar de MNAR, procure **proxies** e documente a incerteza.

# How to Diagnose Missing Data Mechanisms

| Mechanism | Key Idea | Visual Pattern |
|---|---|---|
| MCAR | Missingness independente de todas as variáveis | Dispersão aleatória, correlações ~ 0 |
| MAR | Missingness depende de outra variável observada | Blocos ou clusters ao ordenar por essa variável |
| MNAR | Missingness depende do próprio valor em falta | Padrões indiretos via proxies, evidência fraca |

**Conclusão e Principais Conclusões:**
- As visualizações do MissingData permitem detectar padrões que seriam invisíveis em tabelas.
- MCAR mostra padrões dispersos; MAR mostra concentração quando ordenamos por uma variável observada.
- MNAR é o mais difícil e exige cautela, proxies e transparência sobre incertezas.